In [1]:
!pip -q install -U git+https://github.com/huggingface/transformers accelerate qwen-vl-utils


  ERROR: Error [WinError 2] The system cannot find the file specified while executing command git version

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?


In [12]:
from google.colab import files
uploaded = files.upload()
image_path = next(iter(uploaded.keys()))
image_path


Saving shelf-image.jfif to shelf-image.jfif


'shelf-image.jfif'

# Loading VLM and extracting book titles as JSON

Using Qwen2-VL-2B-Instruct by default, can be switched to 7B by changing the model id if you have enough VRAM

In [4]:
!pip -q install json-repair


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.1 MB/s eta 0:00:00


In [13]:
import json, re
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from json_repair import repair_json

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(MODEL_ID, use_fast=False)

img = Image.open(image_path).convert("RGB")

INSTR = """You are extracting book spine text from a bookshelf photo.
Return ONLY valid JSON with this schema:
{
  "books": [
    {"title": "...", "author": null or "...", "confidence": 0.0-1.0}
  ]
}
Rules:
- Prefer full titles; omit words you are unsure about.
- If author is not visible, set author to null.
- Output max 30 items.
"""

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": INSTR},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(model.device)

def parse_model_json(raw: str):
    # extract first JSON like object
    m = re.search(r"\{.*\}", raw, flags=re.S)
    if not m:
        raise ValueError("No JSON object found. Raw output:\n" + raw)

    blob = m.group(0)

    # strict parse first
    try:
        return json.loads(blob)
    except json.JSONDecodeError:
        # repair common LLM JSON issues like missing commas, trailing commas, bad quotes, etc.
        fixed = repair_json(blob)
        return json.loads(fixed)

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False,
        temperature=0.0,
    )

# trimming prompt tokens as it is a recommended for Qwen2-VL
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

raw = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0].strip()

data = parse_model_json(raw)
candidates = data.get("books", [])
candidates[:5], len(candidates)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

([{'title': 'The Lost Girls of Paris',
   'author': 'Pam Jenoff',
   'confidence': 0.0},
  {'title': 'The Bent Collector', 'author': 'Pam Jenoff', 'confidence': 0.0},
  {'title': "The Diplomat's Wife", 'author': 'Pam Jenoff', 'confidence': 0.0},
  {'title': 'The Book Woman of Troublesome Creek',
   'author': 'Pam Jenoff',
   'confidence': 0.0},
  {'title': 'The Dragon Bones', 'author': 'Lisa See', 'confidence': 0.0}],
 18)

In [7]:
!pip -q install rapidfuzz requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.6 MB/s eta 0:00:00


Cleaning book candidates and normalization

In [14]:
from rapidfuzz import fuzz

def normalize_title(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9\s:,'-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

clean = []
for b in candidates:
    t = (b.get("title") or "").strip()
    if len(t) >= 3:
        clean.append({
            "title": t,
            "title_norm": normalize_title(t),
            "author": b.get("author"),
            "confidence": float(b.get("confidence") or 0.0),
        })

clean = sorted(clean, key=lambda x: x["confidence"], reverse=True)
clean[:10]


[{'title': 'The Lost Girls of Paris',
  'title_norm': 'the lost girls of paris',
  'author': 'Pam Jenoff',
  'confidence': 0.0},
 {'title': 'The Bent Collector',
  'title_norm': 'the bent collector',
  'author': 'Pam Jenoff',
  'confidence': 0.0},
 {'title': "The Diplomat's Wife",
  'title_norm': "the diplomat's wife",
  'author': 'Pam Jenoff',
  'confidence': 0.0},
 {'title': 'The Book Woman of Troublesome Creek',
  'title_norm': 'the book woman of troublesome creek',
  'author': 'Pam Jenoff',
  'confidence': 0.0},
 {'title': 'The Dragon Bones',
  'title_norm': 'the dragon bones',
  'author': 'Lisa See',
  'confidence': 0.0},
 {'title': 'The Collector',
  'title_norm': 'the collector',
  'author': 'Pam Jenoff',
  'confidence': 0.0},
 {'title': 'The Collector',
  'title_norm': 'the collector',
  'author': 'Pam Jenoff',
  'confidence': 0.0},
 {'title': 'The Collector',
  'title_norm': 'the collector',
  'author': 'Pam Jenoff',
  'confidence': 0.0},
 {'title': 'The Collector',
  'title_n

Resolving book candidates with Google Books API.

In [15]:
import requests

def google_books_search(title, author=None, max_results=5):
    q = f'intitle:"{title}"'
    if author:
        q += f'+inauthor:"{author}"'
    url = "https://www.googleapis.com/books/v1/volumes"
    r = requests.get(url, params={"q": q, "maxResults": max_results, "printType": "books"})
    r.raise_for_status()
    return r.json().get("items", [])

def pick_best_match(title_norm, items):
    best = None
    best_score = -1
    for it in items:
        vi = it.get("volumeInfo", {})
        t = normalize_title(vi.get("title", ""))
        score = fuzz.token_set_ratio(title_norm, t)
        if score > best_score:
            best_score = score
            best = it
    return best, best_score

resolved = []
for c in clean[:30]:
    items = google_books_search(c["title"], c["author"])
    best, score = pick_best_match(c["title_norm"], items)
    if best and score >= 70:
        vi = best.get("volumeInfo", {})
        resolved.append({
            "query_title": c["title"],
            "match_score": score,
            "title": vi.get("title"),
            "authors": vi.get("authors", []),
            "categories": vi.get("categories", []),
            "description": vi.get("description", ""),
            "publishedDate": vi.get("publishedDate", ""),
            "averageRating": vi.get("averageRating", None),
            "ratingsCount": vi.get("ratingsCount", None),
            "infoLink": vi.get("infoLink", None),
        })

len(resolved), resolved[:3]


(3,
 [{'query_title': 'The Lost Girls of Paris',
   'match_score': 100.0,
   'title': 'The Lost Girls of Paris',
   'authors': ['Pam Jenoff'],
   'categories': ['Fiction'],
   'description': "The New York Times bestseller—for fans of All the Light We Cannot See and The Tattooist of Auschwitz! Three women. One daring mission. 1946. One morning while passing through Grand Central Terminal, Grace Healey finds an abandoned suitcase tucked beneath a bench. Inside is a dozen photographs—each of a different woman. Grace soon learns that the suitcase belonged to Eleanor Trigg, leader of a network of female secret agents deployed out of London during the war. Twelve of these women were sent to Occupied Europe as couriers and radio operators to aid the resistance, but they never returned home. Setting out to learn the truth behind the women in the photographs, Grace finds herself drawn to a young mother turned agent named Marie, whose mission overseas reveals a remarkable story of friendship, va

### Please list your reading preferences here

In [10]:
pref = input("Enter your reading preferences (genres/themes/authors/pacing): ").strip()
pref


Enter your reading preferences (genres/themes/authors/pacing): paris


'paris'

Embedding and cosine similarity ranking

In [16]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def book_text(b):
    parts = [
        b.get("title",""),
        " ".join(b.get("authors",[])),
        " ".join(b.get("categories",[])),
        b.get("description","")
    ]
    return "\n".join([p for p in parts if p])

book_texts = [book_text(b) for b in resolved]
book_vecs = embedder.encode(book_texts, normalize_embeddings=True)
pref_vec = embedder.encode([pref], normalize_embeddings=True)[0]

sims = book_vecs @ pref_vec
idx = np.argsort(-sims)

topn = 3
recs = []
for i in idx[:topn]:
    b = resolved[i].copy()
    b["similarity"] = float(sims[i])
    recs.append(b)

recs


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'query_title': 'The Lost Girls of Paris',
  'match_score': 100.0,
  'title': 'The Lost Girls of Paris',
  'authors': ['Pam Jenoff'],
  'categories': ['Fiction'],
  'description': "The New York Times bestseller—for fans of All the Light We Cannot See and The Tattooist of Auschwitz! Three women. One daring mission. 1946. One morning while passing through Grand Central Terminal, Grace Healey finds an abandoned suitcase tucked beneath a bench. Inside is a dozen photographs—each of a different woman. Grace soon learns that the suitcase belonged to Eleanor Trigg, leader of a network of female secret agents deployed out of London during the war. Twelve of these women were sent to Occupied Europe as couriers and radio operators to aid the resistance, but they never returned home. Setting out to learn the truth behind the women in the photographs, Grace finds herself drawn to a young mother turned agent named Marie, whose mission overseas reveals a remarkable story of friendship, valor and be

# Recommendations for users view

In [17]:
for k, b in enumerate(recs, 1):
    print(f"\n{k}) {b['title']} — {', '.join(b.get('authors', []))}")
    print(f"   similarity: {b['similarity']:.3f} | match_score: {b['match_score']}")
    if b.get("categories"):
        print(f"   categories: {', '.join(b['categories'][:4])}")
    if b.get("averageRating") is not None:
        print(f"   rating: {b['averageRating']} ({b.get('ratingsCount','?')} ratings)")
    if b.get("infoLink"):
        print(f"   link: {b['infoLink']}")



1) The Lost Girls of Paris — Pam Jenoff
   similarity: 0.211 | match_score: 100.0
   categories: Fiction
   link: https://play.google.com/store/books/details?id=nBZUDwAAQBAJ&source=gbs_api

2) The Diplomat's Wife — Pam Jenoff
   similarity: 0.129 | match_score: 100.0
   categories: Aircraft accidents
   link: http://books.google.com/books?id=v-jqHgAACAAJ&dq=intitle:%22The+Diplomat%27s+Wife%22%2Binauthor:%22Pam+Jenoff%22&hl=&as_pt=BOOKS&source=gbs_api

3) Dragon Bones — Lisa McMann
   similarity: -0.034 | match_score: 100.0
   categories: Juvenile Fiction
   rating: 5 (1 ratings)
   link: https://play.google.com/store/books/details?id=BPUxDwAAQBAJ&source=gbs_api
